## ការបញ្ចូល (Embeddings)

នៅក្នុងឧទាហរណ៍មុនរបស់យើង យើងបានដំណើរការលើវ៉ិកទ័រកញ្ចប់ពាក្យដែលមានវិមាត្រខ្ពស់ជាមួយប្រវែង `vocab_size` ហើយយើងបានបម្លែងយ៉ាងច្បាស់លាស់ពីវ៉ិកទ័រតំណាងទីតាំងវិមាត្រទាបទៅជាការតំណាង one-hot ដែលមានភាពរាយស្រឡាយ។ ការតំណាង one-hot នេះមិនប្រសើរដល់មេម៉ូរីទេ។ លើសពីនេះទៀត ពាក្យនីមួយៗត្រូវបានគេព្យាយាមឲ្យមានភាពឯករាជ្យពីគ្នា ដូច្នេះវ៉ិកទ័រដែលបាន encode ជា one-hot មិនបង្ហាញអត្ថន័យស្រដៀងគ្នារវាងពាក្យទេ។

នៅក្នុងឯកតានេះ យើងនឹងបន្តស្វែងយល់អំពីទិន្នន័យ **News AG**។ ដើម្បីចាប់ផ្តើម ចូរយើងផ្ទុកទិន្នន័យ ហើយយកនិយមន័យខ្លះៗពីឯកតាមុន។


In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

ds_train, ds_test = tfds.load('ag_news_subset').values()

### embedding គឺជាអ្វី?

គំនិតនៃ **embedding** គឺដើម្បីតំណាងឱ្យពាក្យដោយប្រើវ៉ិចទ័រចំនួនតិចដែលមានមាតិកាត្រូវគ្នានឹង អត្ថបទនៃពាក្យនោះ។ យើងនឹងពិភាក្សាថាតើធ្វើដើម្បីបង្កើត embedding ដែលមានន័យយ៉ាងដូចម្តេច ប៉ុន្តេលៅពេលនេះ មកគិតថា embedding គឺជា វិធីសាស្រ្តមួយសម្រាប់កាត់បន្ថយវិមាត្ររបស់វ៉ិចទ័រពាក្យ។

ដូចនេះ ស្រទាប់ embedding នេះទទួលពាក្យជាអញ្ញាត និងបង្កើតវ៉ិចទ័រចេញដែលមានវិមាត្រត្រូវបានបញ្ជាក់ដោយ `embedding_size`។ មានន័យថា វាស្រដៀងនឹងស្រទាប់ `Dense` មួយ ប៉ុន្តែប្ដូរពីការទទួលវ៉ិចទ័រចែកបែប one-hot encoded ជា ចំនួនពាក្យ។

ដោយប្រើស្រទាប់ embedding ជាស្រទាប់ដំបូងក្នុងបណ្តាញរបស់យើង យើងអាចប្ដូរពី bag-of-words ទៅម៉ូដែល **embedding bag** ដែលយើងចាប់ផ្តើមផ្លាស់ប្ដូរពីពាក្យនីមួយៗទៅ embedding ផ្ទាល់ខ្លួន រួចបញ្ចូលករណីហាងសម្រាប់ embed ផ្សេងៗទាំងអស់ ឧទាហរណ៍ `sum`, `average` ឬ `max`។

![រូបភាពបង្ហាញពី embedding classifier សម្រាប់ ពាក្យលំដាប់ប្រាំ។](../../../../../translated_images/km/embedding-classifier-example.b77f021a7ee67eee.webp)

បណ្តាញសរសេរ neural network របស់យើងមានស្រទាប់ដូចខាងក្រោម៖

* ស្រទាប់ `TextVectorization` ដែលទទួលខ្សែអក្សរជាអញ្ញាត ហើយបង្កើត tensor នៃលេខ​តំណាង​ពាក្យ។ យើងនឹងបញ្ជាក់ទំហំវាកប៊ូលារីវិធីសាស្រ្តមួយ `vocab_size` ហើយមិនបញ្ជូនពាក្យដែលប្រើប្រាស់តិចតួច។ ទម្រង់ input នឹងមានទំហំនៅ 1 ហើយទម្រង់ output នឹងមានទំហំ $n$ ពីព្រោះយើងនឹងទទួលបាន $n$ token ក្នុងលទ្ធផល ដែលមួយក្នុងនោះមានលេខពី 0 រហូតដល់ `vocab_size`។
* ស្រទាប់ `Embedding` ដែលទទួលលេខ $n$ ហើយបញ្ជ្រាបលេខនីមួយៗទៅជាវ៉ិចទ័រចម្បងដែលមានប្រវែងបញ្ជាក់ (100 ក្នុងឧទាហរណ៍របស់យើង)។ ដូច្នេះ tensor input ដែលមានទំហំ $n$ នឹងបំលែងទៅជា tensor ទំហំ $n\times 100$។
* ស្រទាប់ aggregation ដែលគណនាមធ្យមនៃ tensor នេះរង្វង់ឯកតាដំបូង គឺវានឹងគណនាមធ្យមនៃពាក្យទាំងអស់ $n$ ទៅជាតិចុងក្រោយ។ ដើម្បីអនុវត្តស្រទាប់នេះ យើងនឹងប្រើស្រទាប់ `Lambda` ហើយផ្ទុកមុខងារ ដែលគណនាមធ្យម។ អ្នកនឹងបាន output ទំហំ 100 ដែលជាតំណាងលេខសំរាប់ភាគីសរុបនៃរៀងក្នុង input ទាំងមូល។
* ស្រទាប់ចុងក្រោយ `Dense` classifier រូបភាព linear។


In [3]:
vocab_size = 30000
batch_size = 128

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size,input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,    
    keras.layers.Embedding(vocab_size,100),
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization (TextVec  (None, None)             0         
 torization)                                                     
                                                                 
 embedding (Embedding)       (None, None, 100)         3000000   
                                                                 
 lambda (Lambda)             (None, 100)               0         
                                                                 
 dense (Dense)               (None, 4)                 404       
                                                                 
Total params: 3,000,404
Trainable params: 3,000,404
Non-trainable params: 0
_________________________________________________________________


នៅក្នុងការបោះពុម្ព `summary` នៅជួរឈរដែលមាន **output shape** កម្រិត tensor ដំបូង `None` បណ្ដាប់ទៅនឹងទំហំ minibatch ហើយកម្រិតទីពីរបណ្ដាប់ទៅនឹងប្រវែងនៃលំដាប់ token ។ លំដាប់ token ទាំងអស់ក្នុង minibatch មានប្រវែងខុសគ្នា។ យើងនឹងពិភាក្សា​អំពីវិធីដោះស្រាយវា​នៅក្នុងផ្នែកបន្ទាប់។

ឥឡូវនេះយើងមកហ្វឹកហាត់បណ្ដាញ៖


In [4]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

print("Training vectorizer")
vectorizer.adapt(ds_train.take(500).map(extract_text))

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

Training vectorizer
938/938 [==============================] - 20s 20ms/step - loss: 0.7891 - acc: 0.8155 - val_loss: 0.4470 - val_acc: 0.8642


> **សម្គាល់** ថា​យើងកំពុងបង្កើតម៉ូឌែលវ៉ិចទ័រជាមួយនឹងទិន្នន័យជាផ្នែកមួយ។ វាត្រូវបានធ្វើឡើងដើម្បីបន្ថែមល្បឿនដំណើរការ ហើយវាអាចនាំឲ្យមានស្ថានភាពដែលមិនមានតួអក្សរទាំងអស់ពីអត្ថបទរបស់យើងក្នុងវាគាគារទេ។ ក្នុងករណីនេះ តួអក្សរទាំងនោះនឹងត្រូវបានមិនគិតជាប់ណាមួយ ដែលអាចនាំឲ្យមានភាពត្រឹមត្រូវតិចតួចទាបជាងមុន។ ទោះជាយ៉ាងណា នៅក្នុងជីវិតពិត ផ្នែកមួយនៃអត្ថបទភាគច្រើនផ្តល់នូវការប៉ាន់ប្រមាណវាគាគារដែលល្អ។


### ដោះស្រាយទំហំតួអក្សរបម្រែបម្រួល

យើងមកយល់ពីរបៀបដែលការបណ្តុះបណ្តាលកើតឡើងក្នុង minibatches។ ក្នុងឧទាហរណ៍ខាងលើ, តួអង់ត្រីមានវិមាត្រតែមួយ, ហើយយើងប្រើ minibatches យ៉ាងយឺត 128 ដូច្នេះទំហំពិតនៃតួអង់ត្រាជា $128 \times 1$។ ទោះបីជារួចហើយ, ចំនួនតួអក្សរនៅក្នុងប្រយោគនីមួយៗគឺខុសគ្នា។ ប្រសិនបើយើងអនុវត្តស្រទាប់ `TextVectorization` ទៅលើបញ្ចូលតែមួយ, ចំនួនតួអក្សរដែលត្រូវបានត្រឡប់មកនឹងខុសគ្នា, អាស្រ័យទៅលើរបៀបដែលអត្ថបទត្រូវបានបំបែកតួអក្សរ៖


In [5]:
print(vectorizer('Hello, world!'))
print(vectorizer('I am glad to meet you!'))

tf.Tensor([ 1 45], shape=(2,), dtype=int64)
tf.Tensor([ 112 1271    1    3 1747  158], shape=(6,), dtype=int64)


យ៉ាងไรก็ตาม ពេលយើងអនុវត្តកម្មវិធីវ៉ិចទ័រជូនលំដាប់ច្រើន វាត្រូវតែបង្កើតទិន្ស័រដែលមានរូបរាងជា​មូរដេដែក ដូច្នេះវាបំពេញធាតុដែលមិនបានប្រើដោយស្លាក PAD (ដែលក្នុងករណីរបស់យើងគឺលេខសូន្យ):


In [6]:
vectorizer(['Hello, world!','I am glad to meet you!'])

<tf.Tensor: shape=(2, 6), dtype=int64, numpy=
array([[   1,   45,    0,    0,    0,    0],
       [ 112, 1271,    1,    3, 1747,  158]], dtype=int64)>

នៅទីនេះ​យើង​អាច​មើលឃើញ​ការ​បញ្ចូលៈ


In [7]:
model.layers[1](vectorizer(['Hello, world!','I am glad to meet you!'])).numpy()

array([[[ 1.53059261e-02,  6.80514947e-02,  3.14026810e-02, ...,
         -8.92002955e-02,  1.52911525e-04, -5.65562584e-02],
        [ 2.57456154e-01,  2.79364467e-01, -2.03605562e-01, ...,
         -2.07474351e-01,  8.31158683e-02, -2.03911960e-01],
        [ 3.98201384e-02, -8.03454965e-03,  2.39790026e-02, ...,
         -7.18549127e-04,  2.66963355e-02, -4.30646613e-02],
        [ 3.98201384e-02, -8.03454965e-03,  2.39790026e-02, ...,
         -7.18549127e-04,  2.66963355e-02, -4.30646613e-02],
        [ 3.98201384e-02, -8.03454965e-03,  2.39790026e-02, ...,
         -7.18549127e-04,  2.66963355e-02, -4.30646613e-02],
        [ 3.98201384e-02, -8.03454965e-03,  2.39790026e-02, ...,
         -7.18549127e-04,  2.66963355e-02, -4.30646613e-02]],

       [[ 1.89674050e-01,  2.61548996e-01, -3.67433839e-02, ...,
         -2.07366899e-01, -1.05442435e-01, -2.36952081e-01],
        [ 6.16133213e-02,  1.80511594e-01,  9.77298319e-02, ...,
         -5.46628237e-02, -1.07340455e-01, -1.06589

> **កំណត់ចំណាំ**: ដើម្បីកាត់បន្ថយបរិមាណ padding ឲ្យតិចបំផុត នៅក្នុងករណីខ្លះវាមានហេតុផលក្នុងការរៀបចំលំដាប់រាល់លំដាប់ក្នុងប្រមូលទិន្នន័យទៅតាមលំដាប់កើនឡើងនៃកាំង (ឬ ផ្ទាល់ខ្លួន គឺចំនួននៃតូកិន)។ នេះនឹងធានាថា មីនីបៀចនីមួយៗមានលំដាប់ដែលមានប្រវែងស្រដៀងគ្នា។


## Semantic embeddings: Word2Vec

ក្នុងឧទាហរណ៍មុនរបស់យើង ស្រទាប់ embedding យល់ដឹងត្រួតពិនិត្យដើម្បីផែនទីពាក្យទៅតំណាងវ៉ិចទ័រប៉ុន្តែ តំណាងទាំងនេះគ្មានអត្ថន័យសម្របសម្រួលទេ។ វានឹងមានប្រយោជន៍សម្រាប់រៀនតំណាងវ៉ិចទ័រដែលយ៉ាងណាក៏ដោយពាក្យស្រដៀងគ្នា ឬពាក្យប្រើប្រាស់ជំនួសគ្នា ត្រូវតែផ្គូផ្គងទៅនឹងវ៉ិចទ័រដែលនៅជិតគ្នានៅលើបាតុភូតវ៉ិចទ័រខ្លះ (ឧទាហរណ៍ចំរូងអ៊ុយកីឡេដៀន)។

ដើម្បីធ្វើការនេះ យើងចាំបាច់ត្រូវ pretrained ម៉ូដែល embedding របស់យើងលើកំណត់ត្រាអត្ថបទដែលធំជាងគេដោយប្រើបច្ចេកទេសដូចជា [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)។ វាមានមូលដ្ឋានលើស្ថាបត្យកម្មសំខាន់ពីរ ដែលត្រូវបានប្រើក្នុងការផលិតតំណាងចែកចាយនៃពាក្យ៖

 - **Continuous bag-of-words** (CBoW) ដែលយើងបណ្តុះម៉ូដែលដើម្បីបញ្ជាក់ទោលពាក្យមួយពីបរិបទជុំវិញ។ បើដាក់ ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ គោលបំណងម៉ូដែលគឺបញ្ជាក់ទោល $W_0$ ពី $(W_{-2},W_{-1},W_1,W_2)$។
 - **Continuous skip-gram** គឺអាចវិជ្ជមាន CBoW។ ម៉ូដែលប្រើបរិបទជុំវិញនៃពាក្យបរិបទ ដើម្បីបញ្ជាក់ទោលពាក្យបច្ចុប្បន្ន។

CBoW លឿនជាង ហើយបើទោះបីជា skip-gram ជាប្រភេទយឺតជាងក៏ដោយ វាដំណើរការល្អជាងក្នុងការបង្ហាញពាក្យដែលតិចធ្លាប់ប្រើ។

![រូបភាពបង្ហាញពីអាល់ក្គរីធម៍ CBoW និង Skip-Gram ដើម្បីបម្លែងពាក្យទៅជាវ៉ិចទ័រ។](../../../../../translated_images/km/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

ដើម្បីសាកល្បងជាមួយ embedding Word2Vec ដែលបាន pretrained លើកំណត់ត្រា Google News dataset យើងអាចប្រើបណ្ណាល័យ **gensim**។ ខាងក្រោមយើងស្វែងរកពាក្យដែលស្រដៀងគ្នាមានអ្នកទំនងជាមួយពាក្យ 'neural'។

> **Note:** នៅពេលអ្នកបង្កើតវ៉ិចទ័រពាក្យលើកដំបូង ការទាញយកវាអាចចំណាយពេលខ្លះ!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [12]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


យើងអាចដកចេញបេក្ខវិស័យវ៉ិចទ័រពីពាក្យ ផងដែរ ដើម្បីប្រើប្រាស់ក្នុងការបណ្តុះម៉ូដែលចំណាត់ថ្នាក់។ បេក្ខវិស័យមានបន្ទុក ៣០០ ប្រភេទ ប៉ុន្តែក្នុងនេះយើងបង្ហាញតែក្នុង ២០ ប្រភេទដំបូងនៃវ៉ិចទ័រដែលមានភាពច្បាស់៖


In [13]:
w2v['play'][:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

រឿងដ៏អស្ចារ្យអំពីការ​អញ្ចូលទិន្នន័យ​អត្ថន័យ​គឺ​អ្នកអាច​ត្រួតពិនិត្យ​កូដ​វ៉ិកទ័រ​ដោយផ្អែក​លើ​អត្ថន័យ​បាន។ ឧទាហរណ៍ យើង​អាចសួរឱ្យ​រកពាក្យ​មួយដែល​គំនូសវ៉ិកទ័រ​របស់វា​នៅជិត​បំផុត​នឹងពាក្យ *king* និង *woman* ហើយ​នៅឆ្ងាយ​បំផុត​ពីពាក្យ *man*៖


In [14]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

ឧទាហរណ៍ខាងលើប្រើមន្តសិទ្ធិ GenSym ខ្លះៗ ប៉ុន្តែលទ្ធផលមួយដែលជាចម្បងគឺសាមញ្ញណាស់។ រឿងចម្លែកមួយអំពី embeddings គឺអ្នកអាចអនុវត្តប្រតិបត្តិការវ៉ិចទ័រធម្មតានៅលើវ៉ិចទ័រនៃ embeddings ហើយវានឹងបញ្ចេញនូវប្រតិបត្តិការលើ **អត្ថន័យ** ពាក្យ។ ឧទាហរណ៍ខាងលើអាចបង្ហាញជាទម្រង់ប្រតិបត្តិការវ៉ិចទ័រ៖ យើងគណនាវ៉ិចទ័រនៃពាក្យដែលសមរម្យនឹង **KING-MAN+WOMAN** (ប្រតិបត្តិការនៃ `+` និង `-` ត្រូវបានអនុវត្តលើតំណាងវ៉ិចទ័រនៃពាក្យដែលសមរម្យ) ហើយបន្ទាប់មកស្វែងរកពាក្យដែលនៅជិតបំផុតនៅក្នុងវចនានុក្រមទៅវ៉ិចទ័រនោះ៖


In [15]:
# get the vector corresponding to kind-man+woman
qvec = w2v['king']-1.7*w2v['man']+1.7*w2v['woman']
# find the index of the closest embedding vector 
d = np.sum((w2v.vectors-qvec)**2,axis=1)
min_idx = np.argmin(d)
# find the corresponding word
w2v.index_to_key[min_idx]

'queen'

> **ចំណាំ**៖ យើងត្រូវបានបន្ថែមអនុគ្រោះតូចៗទៅកន្លែងវ៉ិកទ័រ *man* និង *woman* - ព្យាយាមដកវាចេញដើម្បីមើលថាអ្វីបានកើតឡើង។

ដើម្បីស្វែងរកវ៉ិកទ័រតំបន់ក្បាលបំផុត យើងប្រើ ឧបករណ៍ TensorFlow ដើម្បីគណនាវ៉ិកទ័ររបស់ចម្ងាយរវាងវ៉ិកទ័ររបស់យើងនិងវ៉ិកទ័រទាំងអស់នៅក្នុងពាក្យសព្ទ ហើយបន្ទាប់មកស្វែងរកសន្ទស្សន៍នៃពាក្យតូចបំផុតដោយប្រើ `argmin`។


ខណៈពេល Word2Vec ดูเหมือนจะเป็นវិធីដ៏អស្ចារ្យសម្រាប់បង្ហាញអត្ថន័យពាក្យ វាមានគុណវិបត្តិជាច្រើន រួមមាន៖

* ទាំងពីរម៉ូដែល CBoW និង skip-gram គឺជាការបញ្ចាក់អembedding **predictive embeddings** ហើយវាទទួលយកតែបរិបទក្នុងសហគមន៍បន្តិចបន្តួចប៉ុណ្ណោះ។ Word2Vec មិនយកប្រយោជន៍ពីបរិបទជាសកលនោះទេ។
* Word2Vec មិនគិតពីរាងកាយពាក្យ **morphology** គឺ ន័យថា អត្ថន័យនៃពាក្យអាចអាស្រ័យលើផ្នែកផ្សេងៗនៃពាក្យ ដូចជាមูลដ្ពៃ។

**FastText** ព្យាយាមដោះស្រាយលំដាប់ទីពីរនេះ ហើយបន្ថែមលើ Word2Vec ដោយរៀនវ៉ិចទ័រដែលតំណាងឲ្យដល់ពាក្យនីមួយៗ និងអក្សរ n-gram ដែលត្រូវបានរកឃើញក្នុងពាក្យនីមួយៗ។ តម្លៃនៃការតំណាងទាំងនេះបន្ទាប់មកត្រូវបានរាប់មធ្យមទៅជា​វ៉ិចទ័រមួយនៅក្នុងជំហានបណ្ដុះបណ្ដាលរៀងរាល់ដង។ ខណៈដែលនេះបន្ថែមកំណត់ឡើងនៃការគណនា​ក្នុងការបណ្ដុះបណ្ដាលជាមុន វាក៏អាចអោយ​ word embeddings ចាប់យកព័ត៌មានក្រោមពាក្យបាន។

វិធីមួយផ្សេងទៀត គឺ **GloVe** ដែលប្រើវិធីផ្សេងទៀតសម្រាប់ word embeddings ដោយផ្អែកលើការបែងចែកម៉ាទ្រីសពាក្យ-បរិបទ។ ជាដំបូង វាសង់ម៉ាទ្រីសដ៏ធំបូកនឹងរាប់ចំនួនករណី​ពីពាក្យក្នុងបរិបទផ្សេងៗ ហើយបន្ទាប់មកវាព្យាយាមតំណាងម៉ាទ្រីសនេះជាទំហំតិចដើម្បីបន្ថយការបាត់បង់ក្នុងការបង្កើតឡើងវិញ។

បណ្ណាល័យ gensim គាំទ្រនូវ word embeddings ទាំងនេះ ហើយអ្នកអាចសាកល្បងជាមួយពួកវា​ដោយផ្លាស់ប្តូរកូដផ្ទុកម៉ូដែលខាងលើ។


## ការប្រើប្រាស់ embeddings ដែលបានបណ្តុះបណ្តាលរួចហើយនៅក្នុង Keras

យើងអាចកែប្រែឧទាហរណ៍ខាងលើ ដើម្បីបំពេញមាត្រដ្ឋានក្នុងស្រទាប់ embedding របស់យើងជាមុនជាមួយ embedding ផ្នែកអត្ថន័យ ដូចជា Word2Vec។ ពាក្យសម្ព័ន្ធនៃ embedding ដែលបានបណ្តុះបណ្តាលរួច និងអត្ថបទខុនតាក់ស៍ មានសក្តានុពលដែលមិនត្រូវគ្នា ដូច្នេះយើងត្រូវជ្រើសរើសមួយ។ នៅទីនេះ យើងស្វែងយល់ពីជម្រើសពីរដែលអាចធ្វើបាន៖ ការប្រើប្រាស់ពាក្យសម្ព័ន្ធ tokenizer និងការប្រើប្រាស់ពាក្យសម្ព័ន្ធពី embedding Word2Vec។

### ការប្រើប្រាស់ពាក្យសម្ព័ន្ធ tokenizer

ពេលប្រើប្រាស់ពាក្យសម្ព័ន្ធ tokenizer ពាក្យខ្លះពីពាក្យសម្ព័ន្ធនឹងមាន embedding Word2Vec ត្រូវគ្នា ហើយពាក្យខ្លះនឹងអាចបាត់បង់។ ដោយសារតែទំហំពាក្យសម្ព័ន្ធរបស់យើងគឺ `vocab_size` និងប្រវែងវ៉ិចទ័រ embedding របស់ Word2Vec គឺ `embed_size` ស្រទាប់ embedding នឹងត្រូវតំណាងដោយមាត្រដ្ឋានទំរង់ `vocab_size`$\times$`embed_size`។ យើងនឹងបំពេញមាត្រដ្ឋាននេះដោយឆ្លងកាត់ពាក្យសម្ព័ន្ធ៖


In [9]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

vocab = vectorizer.get_vocabulary()
W = np.zeros((vocab_size,embed_size))
print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab):
    try:
        W[i] = w2v.get_vector(w)
        found+=1
    except:
        # W[i] = np.random.normal(0.0,0.3,size=(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")

Embedding size: 300
Populating matrix, this will take some time...Done, found 4551 words, 784 words missing


សម្រាប់ពាក្យដែលមិនមាននៅក្នុងវចនានុក្រម Word2Vec យើងអាចទុកវាជា សូន្យ ឬបង្កើតវ៉ិចទ័រដោយចៃដន្យ។

ឥឡូវនេះយើងអាចកំណត់ស្រទាប់ embedding មួយជាមួយទំងន់ដែលបានបណ្តុះរួចហើយ៖


In [10]:
emb = keras.layers.Embedding(vocab_size,embed_size,weights=[W],trainable=False)
model = keras.models.Sequential([
    vectorizer, emb,
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])

ឥឡូវនេះ យើងមកបណ្តុះបណ្តាលម៉ូឌែលរបស់យើង។


In [11]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

938/938 [==============================] - 10s 10ms/step - loss: 1.1075 - acc: 0.7822 - val_loss: 0.9134 - val_acc: 0.8175


> **សម្គាល់**: សូមកត់សម្គាល់ថា យើងកំណត់ `trainable=False` នៅពេលបង្កើត `Embedding` ដែលមានន័យថាយើងមិនបណ្តុះបណ្តាលឡើងវិញស្រទាប់ Embedding នោះទេ។ វាអាចបណ្តាលអោយកម្រិតទំនាក់ទំនងក័រម្លិចបន្តិច ប៉ុន្តែវា прискорює​ ការបណ្តុះបណ្តាល។

### ការប្រើប្រាស់វចនានុក្រម embedding

បញ្ហាមួយជាមួយវិធីសាស្រ្តមុនគឺវចនានុក្រមដែលប្រើនៅក្នុង TextVectorization និង Embedding គឺខុសគ្នា។ ដើម្បីដោះស្រាយបញ្ហានេះ យើងអាចប្រើដំណោះស្រាយដូចខាងក្រោមមួយណាមួយ៖
* បណ្តុះបណ្តាលម៉ូដែល Word2Vec ម្តងទៀតលើវចនានុក្រមរបស់យើង។
* បង្ហាញទិន្នន័យរបស់យើងជាមួយវចនានុក្រមពីម៉ូដែល Word2Vec ដែលបានបណ្តុះក្រោយ។ វចនានុក្រមដែលប្រើសម្រាប់បញ្ចូលទិន្នន័យអាចកំណត់នៅពេលបញ្ចូល។

វិធីសាស្រ្តទីពីរមើលទៅងាយស្រួលជាង ដូចនេះយើងចាប់ផ្តើមអនុវត្តវា។ ជាទីមុន យើងនឹងបង្កើតស្រទាប់ `TextVectorization` ជាមួយវចនានុក្រមដែលបានកំណត់ ដែលយកចេញពី embeddings របស់ Word2Vec ៖


In [12]:
vocab = list(w2v.vocab.keys())
vectorizer = keras.layers.experimental.preprocessing.TextVectorization(input_shape=(1,))
vectorizer.set_vocabulary(vocab)

បណ្ណាល័យ word embeddings របស់ gensim មានមោទនភាពមួយដែលមានប្រយោជន៍ គឺ `get_keras_embeddings` ដែលនឹងបង្កើតស្រទាប់ embeddings នៃ Keras ដែលផ្ទាល់សម្រាប់អ្នកដោយស្វ័យប្រវត្តិ។


In [13]:
model = keras.models.Sequential([
    vectorizer, 
    w2v.get_keras_embedding(train_embeddings=False),
    keras.layers.Lambda(lambda x: tf.reduce_mean(x,axis=1)),
    keras.layers.Dense(4, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'])
model.fit(ds_train.map(tupelize).batch(128),validation_data=ds_test.map(tupelize).batch(128),epochs=5)

Epoch 1/5
938/938 [==============================] - 20s 14ms/step - loss: 1.3377 - acc: 0.4978 - val_loss: 1.2995 - val_acc: 0.5647
Epoch 2/5
938/938 [==============================] - 10s 10ms/step - loss: 1.2587 - acc: 0.5722 - val_loss: 1.2339 - val_acc: 0.5842
Epoch 3/5
938/938 [==============================] - 10s 10ms/step - loss: 1.1980 - acc: 0.5884 - val_loss: 1.1826 - val_acc: 0.5954
Epoch 4/5
938/938 [==============================] - 12s 13ms/step - loss: 1.1503 - acc: 0.6002 - val_loss: 1.1417 - val_acc: 0.6018
Epoch 5/5
938/938 [==============================] - 11s 12ms/step - loss: 1.1120 - acc: 0.6097 - val_loss: 1.1083 - val_acc: 0.6104


មូលហេតុមួយដែលធ្វើឲ្យយើងមិនឃើញភាពត្រឹមត្រូវខ្ពស់ជាងនេះ គឺព្រោះពាក្យជាច្រើនពីឃ្លាសំណុំនៃទិន្នន័យរបស់យើងខាតនៅក្នុងវចនានុក្រម GloVe ដែលបានបណ្តុះបណ្តាលមុន ហើយដូច្នេះពួកវាត្រូវបានអះអាងចោល។ ដើម្បីទប់ស្កាត់បញ្ហានេះ យើងអាចបណ្តុះបណ្តាលការបង្ហាញបែប embeddings មួយផ្ទាល់លើមូលដ្ឋានទិន្នន័យរបស់យើង។


## ពន្យល់ផ្ទៃក្នុងបរិបទ

កំណត់ខ្លាំងមួយនៃការបង្ហាញអត្តសញ្ញាណដែលបានបណ្ដុះបណ្ដាលជាមុនជារូបភាពដូចជា Word2Vec គឺថា ទោះបីត្រូវចាប់អត្ថន័យនៃពាក្យមួយក៏ដោយ ពួកវាមិនអាចបំបែកតាមអត្ថន័យផ្សេងៗបាននោះទេ។ នេះអាចបណ្តាលបញ្ហាក្នុងម៉ូដែលដែលយើងប្រើបន្ទាប់។

ឧទាហរណ៍ពាក្យ 'play' មានអត្ថន័យខុសគ្នានៅក្នុងប្រយោគទាំងពីរនេះ៖
- ខ្ញុំបានទៅចូលរួម **play** នៅក្នុងក្រហម។
- John ចង់ **play** ជាមួយមិត្តភក្តិរបស់គាត់។

ការបង្ហាញអត្តសញ្ញាណដែលបានបណ្ដុះបណ្ដាលជាមុនដែលយើងបានគិតពីនេះ បង្ហាញពីអត្ថន័យទាំងពីរនៃពាក្យ 'play' នៅក្នុងរូបភាពដើម្បីតែមួយ។ ដើម្បីរំដោះកំណត់ខ្លាំងនេះ យើងត្រូវតែបង្កើតការបង្ហាញអត្តសញ្ញាណដែលផ្អែកលើ **ម៉ូដែលភាសា** ដែលត្រូវបានបណ្តុះបណ្តាលលើប័ណ្ណអត្ថបទច្រើន និង *ដឹង* ថាពាក្យអាចត្រូវបានដាក់ក្នុងបរិបទផ្សេងៗគ្នាយ៉ាងដូចម្តេច។ ការពិភាក្សាពីការបង្ហាញអត្តសញ្ញាណផ្ទៃក្នុងបរិបទ នៅក្រៅពីសកម្មភាពសិក្សានេះ ប៉ុន្តែយើងនឹងត្រឡប់មកវិញនៅពេលនិយាយអំពីម៉ូដែលភាសាក្នុងឯកតាបន្ទាប់។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការព្រមាន**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលយើងខំប្រឹងប្រែងដើម្បីឲ្យមានភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយស្វ័យប្រវត្តិក្នុងរហៈពេលខ្លីអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅភាសាតំបន់របស់វា គួរត្រូវបានគេចាត់ទុកជា ប្រភពផ្លូវការជាធរមាន។ សម្រាប់ព័ត៌មានចំរូងចំរាស សូមប្រើការបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែខុសផ្សេងទៀតដែលខ្លួនកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
